# llm_client

> The Claude API client for Phase 10: one structured-output request per turn, an on-disk response cache (the only source of exact reruns), a spending cap shared across processes, and an offline `FakeClient`. Only this module imports `anthropic`, and only when a live client is built.

In [ ]:
#| default_exp llm_client

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import fcntl
import hashlib
import json
import os
from dataclasses import asdict, dataclass
from pathlib import Path
from types import SimpleNamespace

PRICES = {   # $ per million tokens (Haiku: prompts up to 100K tokens), from the pricing page 2026-10-12; re-checked in T9
    'claude-haiku-5-5': dict(input=0.10, output=0.50, cacheWrite=0.125, cacheRead=0.01, checked='2026-10-12'),
    'claude-sonnet-5-5': dict(input=2.00, output=10.00, cacheWrite=2.50, cacheRead=0.10, checked='2026-10-12'),
}
TOKEN_PROFILE = {   # per call: (uncached input, cache-read input, output) tokens, measured in the pilot (T7, 2026-10-12)
    ('claude-haiku-5-5', 'forage'): (840, 1824, 433), ('claude-haiku-5-5', 'board'): (208, 1771, 901),
    ('claude-sonnet-5-5', 'forage'): (840, 1824, 433), ('claude-sonnet-5-5', 'board'): (214, 1743, 805),   # forage: unused
}
USAGE = ('input_tokens', 'cache_creation_input_tokens', 'cache_read_input_tokens', 'output_tokens')


class BudgetExceeded(RuntimeError):
    pass


def loadEnvFile(path=None):
    """Put KEY=VALUE lines of a git-ignored env file (default: `.env.local` at the repo root) into `os.environ`,
    without overriding variables already set and skipping empty values; returns the names set (never the values)."""
    if path is None:
        root = Path.cwd()
        while not (root / 'pyproject.toml').exists() and root != root.parent:
            root = root.parent
        path = root / '.env.local'
    path, names = Path(path), []
    for line in path.read_text().splitlines() if path.exists() else []:
        k, sep, v = line.strip().removeprefix('export ').partition('=')
        k, v = k.strip(), v.strip().strip('"').strip("'")
        if sep and k and not k.startswith('#') and v and k not in os.environ:
            os.environ[k] = v
            names.append(k)
    return names


def costOf(usage, price):
    "Dollars for one reply's usage."
    return (usage.get('input_tokens', 0) * price['input'] + usage.get('cache_creation_input_tokens', 0) * price['cacheWrite']
            + usage.get('cache_read_input_tokens', 0) * price['cacheRead'] + usage.get('output_tokens', 0) * price['output']) / 1e6


def estimateCost(model, task, agents, turns, prices=PRICES, profile=TOKEN_PROFILE):
    "Expected dollars for `agents` x `turns` calls from the token profile."
    i, r, o = profile[(model, task)]
    p = prices[model]
    return agents * turns * (i * p['input'] + r * p['cacheRead'] + o * p['output']) / 1e6


def requestKey(req):
    return hashlib.sha256(json.dumps(req, sort_keys=True, separators=(',', ':')).encode()).hexdigest()


@dataclass
class LLMReply:
    text: str
    parsed: object
    stopReason: str
    refusal: object
    usage: dict
    cost: float          # the cost of the original live call (also on cache hits, so rows do not depend on the cache)
    cached: bool = False

    def asDict(self):
        d = asdict(self)
        d.pop('cached')
        return d

    @classmethod
    def fromResponse(cls, resp, price):
        text = next((b.text for b in resp.content if getattr(b, 'type', None) == 'text'), '')
        stop, refusal, parsed = resp.stop_reason, None, None
        if stop == 'refusal':
            refusal = getattr(getattr(resp, 'stop_details', None), 'category', None) or 'unknown'
        elif stop != 'max_tokens':
            try:
                parsed = json.loads(text)
            except (ValueError, TypeError):
                parsed = None
        usage = {k: int(getattr(resp.usage, k, 0) or 0) for k in USAGE}
        return cls(text, parsed, stop, refusal, usage, costOf(usage, price))


class ResponseCache:
    "One JSON file per request key; written atomically; holds replies only (no headers, no credentials)."
    def __init__(self, root):
        self.root = Path(root)

    def _path(self, key):
        return self.root / key[:2] / f'{key[2:]}.json'

    def get(self, key):
        p = self._path(key)
        return json.loads(p.read_text()) if p.exists() else None

    def put(self, key, value):
        "Store unless present (first writer wins, across processes); returns the stored value."
        p = self._path(key)
        p.parent.mkdir(parents=True, exist_ok=True)
        tmp = p.with_name(f'{p.name}.{os.getpid()}.tmp')
        tmp.write_text(json.dumps(value, sort_keys=True))
        try:
            os.link(tmp, p)
            return json.loads(tmp.read_text())
        except FileExistsError:
            return json.loads(p.read_text())
        finally:
            tmp.unlink()


class Budget:
    "A spending cap shared by processes through an append-only ledger under a file lock."
    def __init__(self, cap, ledger):
        self.cap, self.ledger = float(cap), Path(ledger)

    def spent(self):
        if not self.ledger.exists():
            return 0.0
        with open(self.ledger) as f:
            fcntl.flock(f, fcntl.LOCK_SH)
            total = sum(json.loads(line)['cost'] for line in f if line.strip())
            fcntl.flock(f, fcntl.LOCK_UN)
        return total

    def remaining(self):
        return self.cap - self.spent()

    def check(self, need=0.0):
        left = self.remaining()
        if left <= 0 or need > left:
            raise BudgetExceeded(f'need ${need:.4f}, ${left:.4f} left of the ${self.cap:.2f} cap')

    def add(self, cost, note=''):
        self.ledger.parent.mkdir(parents=True, exist_ok=True)
        with open(self.ledger, 'a') as f:
            fcntl.flock(f, fcntl.LOCK_EX)
            f.write(json.dumps(dict(cost=float(cost), note=note)) + '\n')
            fcntl.flock(f, fcntl.LOCK_UN)


class LLMClient:
    """One request per turn: cached system prompt, one user message, structured-output schema and effort in
    `output_config`; no sampling parameters, no refusal fallback. Replies are cached by request; only live calls are
    charged to the budget."""
    def __init__(self, model, effort='low', maxTokens=2000, cache=None, budget=None, prices=PRICES, client=None):
        if client is None:
            import anthropic                                     # optional extra: pip install -e ".[llm]"
            client = anthropic.Anthropic(max_retries=6)          # credentials from the environment / SDK profile
        self.model, self.effort, self.maxTokens, self.client = model, effort, int(maxTokens), client
        self.cache, self.budget, self.prices = cache, budget, prices
        self.calls = self.hits = 0
        self.pending = 0.0

    def reserve(self, estimate):
        "A run's expected cost: checked against what is left before its first live call (a fully cached rerun is never refused)."
        self.pending = float(estimate)

    def request(self, system, blocks, schema):
        return dict(model=self.model, max_tokens=self.maxTokens,
                    system=[{'type': 'text', 'text': system, 'cache_control': {'type': 'ephemeral'}}],
                    messages=[{'role': 'user', 'content': blocks}],
                    output_config={'effort': self.effort, 'format': {'type': 'json_schema', 'schema': schema}})

    def complete(self, system, blocks, schema):
        req = self.request(system, blocks, schema)
        key = requestKey(req)
        hit = self.cache.get(key) if self.cache is not None else None
        if hit is not None:
            self.hits += 1
            return LLMReply(**hit, cached=True)
        if self.budget is not None:
            self.budget.check(self.pending)
            self.pending = 0.0
        reply = LLMReply.fromResponse(self.client.messages.create(**req), self.prices[self.model])
        self.calls += 1
        if self.cache is not None:          # a concurrent identical request may have stored its reply first: use that
            reply = LLMReply(**self.cache.put(key, reply.asDict()))
        if self.budget is not None:
            self.budget.add(reply.cost, self.model)
        return reply


class FakeClient:
    """Offline stand-in with the `messages.create` surface: `script(request) -> dict(text, stop, refusal, usage)` or a
    list of such dicts (or plain texts) used in turn. Usage defaults to a deterministic size estimate."""
    def __init__(self, script):
        self.script, self.requests = script, []
        self.messages = SimpleNamespace(create=self._create)

    def _create(self, **req):
        self.requests.append(req)
        r = self.script(req) if callable(self.script) else self.script[(len(self.requests) - 1) % len(self.script)]
        r = dict(text=r) if isinstance(r, str) else dict(r)
        text = r.get('text', '')
        usage = {**dict(input_tokens=len(json.dumps(req)) // 4, cache_creation_input_tokens=0, cache_read_input_tokens=0,
                        output_tokens=max(1, len(text) // 4)), **r.get('usage', {})}
        details = SimpleNamespace(category=r.get('refusal')) if r.get('stop') == 'refusal' else None
        return SimpleNamespace(content=[SimpleNamespace(type='text', text=text)], stop_reason=r.get('stop', 'end_turn'),
                               stop_details=details, usage=SimpleNamespace(**usage))

## Tests

In [ ]:
import json, os, re, tempfile
from pathlib import Path
from explRL.llm_client import *
from explRL.llm_client import costOf, requestKey

schema = {'type': 'object', 'properties': {'action': {'type': 'string'}}, 'required': ['action'], 'additionalProperties': False}
tmp = Path(tempfile.mkdtemp())
i0, r0, o0 = TOKEN_PROFILE[('claude-haiku-5-5', 'forage')]
fake = FakeClient([dict(text='{"action": "stay"}', usage=dict(input_tokens=i0, cache_read_input_tokens=r0, output_tokens=o0))])
c = LLMClient('claude-haiku-5-5', cache=ResponseCache(tmp / 'cache'), budget=Budget(1.0, tmp / 'spend.jsonl'), client=fake)
blocks = [{'type': 'text', 'text': 'turn 1'}]
r1 = c.complete('rules', blocks, schema)
assert r1.parsed == {'action': 'stay'} and not r1.cached and c.calls == 1
assert abs(r1.cost - estimateCost('claude-haiku-5-5', 'forage', 1, 1)) < 1e-12   # the profile prices like usage
# a cache hit replays the reply, makes no request and charges nothing
r2 = c.complete('rules', blocks, schema)
assert r2.cached and r2.parsed == r1.parsed and r2.cost == r1.cost and len(fake.requests) == 1 and c.hits == 1
assert abs(c.budget.spent() - r1.cost) < 1e-12
# the request: no sampling parameters, cached system prompt, schema and effort in output_config
req = fake.requests[0]
assert set(req) == {'model', 'max_tokens', 'system', 'messages', 'output_config'}
assert req['system'][0]['cache_control'] == {'type': 'ephemeral'} and req['output_config']['effort'] == 'low'
assert req['output_config']['format'] == {'type': 'json_schema', 'schema': schema}
# first writer wins: a later reply for the same key is not stored, and the stored one is returned
rc = ResponseCache(tmp / 'race')
assert rc.put('ab12', {'x': 1}) == {'x': 1} and rc.put('ab12', {'x': 2}) == {'x': 1} and rc.get('ab12') == {'x': 1}
assert not list((tmp / 'race').rglob('*.tmp'))
# a different request misses the cache
c.complete('rules', [{'type': 'text', 'text': 'turn 2'}], schema)
assert len(fake.requests) == 2

# refusal and max_tokens stops give no parsed action
for stop, ref in [('refusal', 'cyber'), ('max_tokens', None)]:
    fc = FakeClient([dict(text='{"action": "go"', stop=stop, refusal=ref)])
    r = LLMClient('claude-haiku-5-5', client=fc).complete('s', blocks, schema)
    assert r.parsed is None and r.stopReason == stop and r.refusal == ref
assert LLMClient('claude-haiku-5-5', client=FakeClient(['not json'])).complete('s', blocks, schema).parsed is None

# the cap: a request is refused once the recorded spend reaches it
b = Budget(0.001, tmp / 'cap.jsonl')
cc = LLMClient('claude-haiku-5-5', budget=b, client=FakeClient([dict(text='{}', usage=dict(input_tokens=10000))]))
cc.complete('s', blocks, schema)                    # $0.001: reaches the cap exactly
try:
    cc.complete('s', [{'type': 'text', 'text': 'x'}], schema); raise AssertionError('cap not enforced')
except BudgetExceeded:
    pass
assert len(cc.client.requests) == 1
try:
    Budget(1.0, tmp / 'est.jsonl').check(need=2.0); raise AssertionError
except BudgetExceeded:
    pass
# a run whose estimate exceeds what is left is refused at its first live call, but a cached rerun is served
c.reserve(5.0)
assert c.complete('rules', blocks, schema).cached
try:
    c.complete('rules', [{'type': 'text', 'text': 'new'}], schema); raise AssertionError('estimate not checked')
except BudgetExceeded:
    pass
c.reserve(0.01)
c.complete('rules', [{'type': 'text', 'text': 'new'}], schema)
assert c.pending == 0.0

# costs and estimates
assert costOf(dict(input_tokens=10**6), PRICES['claude-sonnet-5-5']) == 2.0
assert abs(estimateCost('claude-haiku-5-5', 'board', 4, 50, profile={('claude-haiku-5-5', 'board'): (1000, 2000, 300)})
           - 4 * 50 * (1000 * .10 + 2000 * .01 + 300 * .50) / 1e6) < 1e-12
assert requestKey(dict(a=1, b=2)) == requestKey(dict(b=2, a=1))

# secrets: nothing key-like is ever written to the cache or the ledger
written = ''.join(p.read_text() for p in tmp.rglob('*') if p.is_file())
assert not re.search(r'sk-ant-|api[_-]?key|x-api-key|authorization', written, re.I)

# the env-file loader: sets missing, non-empty variables only, and reports names, not values
env = tmp / '.env.local'
env.write_text('# comment\nexport EXPLRL_T1="abc"\nEXPLRL_T2=\nEXPLRL_T3=xyz\nnot a line\n')
os.environ['EXPLRL_T3'] = 'kept'
assert loadEnvFile(env) == ['EXPLRL_T1'] and os.environ['EXPLRL_T1'] == 'abc' and os.environ['EXPLRL_T3'] == 'kept'
assert 'EXPLRL_T2' not in os.environ and loadEnvFile(tmp / 'missing') == []
for k in ('EXPLRL_T1', 'EXPLRL_T3'):
    del os.environ[k]

# the package imports and runs offline without the SDK
import subprocess, sys
code = ("import sys; sys.modules['anthropic'] = None\n"
        "import explRL, explRL.llm_client as m\n"
        "m.LLMClient('claude-haiku-5-5', client=m.FakeClient(['{}'])).complete('s', [], {})\n"
        "try:\n    m.LLMClient('claude-haiku-5-5'); raise SystemExit('imported')\nexcept ImportError: print('ok')")
out = subprocess.run([sys.executable, '-c', code], capture_output=True, text=True)
assert out.stdout.strip() == 'ok', out.stderr

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()